# DS2002 · SQL Fundamentals in Notebooks

**Lecture — 2026-09-07 · Fall 2026**  
**Class time:** 45 minutes

---

## Before we write SQL

A database is just a structured way to store related facts so you can ask better questions later.

A spreadsheet usually starts as one big grid. A database usually splits that grid into tables, with each table holding one kind of thing: people, orders, songs, plays, whatever the system needs to remember.

That sounds like extra work at first. It is. But it makes the data easier to update, easier to search, and much harder to quietly break.

### Tables, rows, and columns

A **table** is one collection of records. Each **row** is one record. Each **column** is one attribute about that record.

If you have a `tracks` table, one row might be one song, and the columns might be `title`, `genre`, `seconds`, and `artist_id`.

```text
tracks
+----------+-----------+------------+---------+-----------+
| track_id | title     | genre      | seconds | artist_id |
+----------+-----------+------------+---------+-----------+
| 10       | Skyline   | Pop        | 201     | 1         |
| 13       | Aurora    | Electronic | 300     | 3         |
+----------+-----------+------------+---------+-----------+
```

One row = one track. That is the question to keep asking as you read or build a table.

The important habit is to ask: what does one row represent in this table? If you can answer that clearly, the table is probably in decent shape.

### What a query does

A **query** is just a question written in SQL.

Sometimes the question is simple: "show me all the tracks." Sometimes it is more specific: "which artist had the most plays last week?" The point of SQL is that you describe the result you want, and the database works out how to produce it.

```text
tracks table
    |
    | WHERE seconds > 200
    v
matching rows
    |
    | SELECT title, seconds
    v
smaller result
    |
    | ORDER BY seconds DESC
    v
final answer
```

Most of the time you are doing one of four things: selecting rows, choosing columns, sorting results, or combining tables.

### Why people normalize data

"Normal form" is database language for keeping data organized so the same fact is not copied all over the place.

If an artist name lives in one place, you correct it once. If it is repeated in five tables or ten thousand rows, every correction becomes cleanup.

```text
One big sheet
play_id | user | track_title | artist_name | artist_country
100     | ava  | Skyline     | Nova Waves  | US
101     | ben  | Skyline     | Nova Waves  | US
102     | ava  | Aurora      | Kestrel     | UK

Split into tables
artists(artist_id, name, country)
tracks(track_id, title, artist_id)
plays(play_id, track_id, user)
```

You do not need to memorize every normal form today. The useful idea is simpler: keep one fact in one place when you can, and connect tables with keys instead of duplication.

## Why a database and not another spreadsheet

Everything so far has been one table at a time. Real data almost never arrives that way. A streaming service does not have a spreadsheet of "plays with all the artist information copied in" — it has a table of artists, a table of tracks, and a table of plays, each storing a fact exactly once.

That structure exists to prevent a specific problem. If the artist's country were copied into every play record, then correcting one typo means finding and fixing it in eleven thousand rows, and you will miss some. Store it once, refer to it everywhere.

SQL is how you ask questions of data shaped like that. It is **declarative**: you describe the result you want, and the database figures out how to get it. That is a real shift from the loops you have been writing.

We use **SQLite**, which is built into Python. No server, no install, no credentials.

### The database we will use all week

Three tables: artists, the tracks they made, and every time somebody pressed play. Run this once. `q()` runs a query and hands back a DataFrame so results render as tables.

In [2]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('tables ready')

tables ready


### Step one, always: what is actually in here?

Never write a query against a database you have not looked at. Two commands tell you everything you need before you start guessing at column names.

In [3]:
q("SELECT name FROM sqlite_master WHERE type='table'")

,name
0,artists
1,tracks
2,plays


In [4]:
q('PRAGMA table_info(tracks)')

,cid,name,type,notnull,dflt_value,pk
0,0,track_id,INTEGER,0,None,1
1,1,title,TEXT,0,None,0
2,2,artist_id,INTEGER,0,None,0
3,3,genre,TEXT,0,None,0
4,4,seconds,INTEGER,0,None,0


`PRAGMA table_info` gives you the column names, their declared types, and whether nulls are allowed. Two minutes here saves you twenty minutes of `KeyError`.

### The clause order, and the order things actually happen

You **write** a query in this order:

```sql
SELECT   ...   -- which columns you want back
FROM     ...   -- which table
WHERE    ...   -- which rows to keep
GROUP BY ...   -- how to bucket them
HAVING   ...   -- which buckets to keep
ORDER BY ...   -- how to sort the result
LIMIT    ...   -- how many rows to show
```

The database **runs** it in a different order: `FROM` -> `WHERE` -> `GROUP BY` -> `HAVING` -> `SELECT` -> `ORDER BY` -> `LIMIT`.

That gap explains an error you are guaranteed to hit. `SELECT` happens *after* `WHERE`, so a column alias you invent in `SELECT` does not exist yet when `WHERE` runs. This fails:

```sql
SELECT seconds / 60 AS minutes FROM tracks WHERE minutes > 4   -- no such column
```

`ORDER BY` runs after `SELECT`, so sorting by that same alias works fine. Keep the execution order in mind and the error messages stop being mysterious.

In [5]:
# Filter, then sort. Note the alias works in ORDER BY but would fail in WHERE.
q('''
SELECT title, genre, seconds, seconds / 60.0 AS minutes
FROM tracks
WHERE seconds > 200
ORDER BY minutes DESC
''')

,title,genre,seconds,minutes
0,Aurora,Electronic,300,5.000000
1,Nightfall,Electronic,275,4.583333
2,Undertow,Pop,240,4.000000
3,Ridgeline,Folk,225,3.750000
4,Sol,Latin,210,3.500000
5,Skyline,Pop,201,3.350000


### Filtering beyond equals

Four operators cover most real filtering. `IN` for a set, `BETWEEN` for a range, `LIKE` for text patterns (`%` matches any number of characters), and `AND`/`OR` to combine.

In [6]:
q('''
SELECT title, genre, seconds
FROM tracks
WHERE genre IN ('Pop', 'Folk')
  AND seconds BETWEEN 180 AND 230
ORDER BY genre, seconds
''')

,title,genre,seconds
0,Foothills,Folk,185
1,Coastline,Folk,199
2,Ridgeline,Folk,225
3,Skyline,Pop,201


**Notes:**

LIKE-- for text


In [7]:
# LIKE for pattern matching. Case-insensitive in SQLite for ASCII.
q("SELECT title FROM tracks WHERE title LIKE '%line%'")

,title
0,Skyline
1,Coastline
2,Ridgeline


### The NULL trap

One track in this database has no genre recorded. `NULL` does not mean zero and it does not mean empty string — it means *unknown*, and comparing anything to unknown gives unknown, not true. So this returns nothing at all:

In [8]:
q('SELECT title, genre FROM tracks WHERE genre = NULL')   # always empty

,title,genre


You have to ask with `IS NULL` instead:

In [9]:
q('SELECT track_id, title, genre FROM tracks WHERE genre IS NULL')

,track_id,title,genre
0,18,Untitled Demo,None


The same logic bites in reverse. `WHERE genre != 'Pop'` also drops the NULL row, because "unknown is not Pop" is itself unknown. If you want it, you have to say so.

In [10]:
print('genre != Pop      ->', len(q("SELECT * FROM tracks WHERE genre != 'Pop'")), 'rows')
print('that, or NULL too ->',
      len(q("SELECT * FROM tracks WHERE genre != 'Pop' OR genre IS NULL")), 'rows')

genre != Pop      -> 6 rows
that, or NULL too -> 7 rows


### Aggregates and GROUP BY

`GROUP BY` collapses rows into buckets, and the aggregate functions summarize each bucket. Anything in `SELECT` that is not aggregated has to be in `GROUP BY`.

In [16]:
#
q('SELECT * from plays')

,play_id,track_id,user,played_on
0,100,10,ava,2026-09-01
1,101,10,ben,2026-09-01
2,102,13,ava,2026-09-02
3,103,13,cara,2026-09-02
4,104,14,ben,2026-09-03
5,105,12,ava,2026-09-03
6,106,15,dan,2026-09-04
7,107,10,cara,2026-09-04
8,108,13,dan,2026-09-05
9,109,16,ava,2026-09-05


In [11]:
q('''
SELECT track_id, COUNT(*) AS plays
FROM plays
GROUP BY track_id --cluster all plays by track
ORDER BY plays DESC
''')

,track_id,plays
0,13,3
1,10,3
2,16,1
3,15,1
4,14,1
5,12,1
6,11,1


Now the distinction that quietly produces wrong numbers in reports: `COUNT(*)` counts **rows**, `COUNT(column)` counts **non-null values** in that column, and `COUNT(DISTINCT column)` counts unique ones.

column counts

dinstinc-- keep info






In [12]:
q('''
SELECT COUNT(*)               AS total_tracks,
       COUNT(genre)           AS tracks_with_a_genre,
       COUNT(DISTINCT genre)  AS distinct_genres
FROM tracks
''')

,total_tracks,tracks_with_a_genre,distinct_genres
0,9,8,4


Nine tracks, eight with a genre, three distinct genres. If someone hands you "we have 8 tracks" and you expected 9, this is usually why.

### HAVING is WHERE for groups

`WHERE` filters rows before grouping. `HAVING` filters the groups after. You cannot put an aggregate in `WHERE`, because when `WHERE` runs the groups do not exist yet.

In [13]:
#filtering after doing group -- HAVING COUNT(*) AS plays
q('''
SELECT track_id, COUNT(*) AS plays
FROM plays
GROUP BY track_id
HAVING COUNT(*) >= 2
ORDER BY plays DESC
''')

,track_id,plays
0,13,3
1,10,3


Read that as: bucket the plays by track, keep only the buckets with at least two plays. Putting `COUNT(*) >= 2` in `WHERE` instead would be an error, and it is the most common SQL mistake in this course.

### Practice 1 — how many people actually listened?

Eleven plays, but fewer humans. Write one query returning the number of distinct users.

In [14]:
q('''

''')

TypeError: 'NoneType' object is not iterable

### Practice 2 — the long tracks

Every track over four minutes, showing title and length in minutes rounded to one decimal, longest first. Remember where an alias can and cannot be used.

In [ ]:
q('''

''')

### Practice 3 — average length per genre

Average track length by genre, only for genres with more than one track, longest average first. What happens to the NULL genre, and is that what you want?

In [ ]:
q('''

''')

### Wednesday

Everything today used one table at a time. Wednesday we connect them — which is where these three tables start answering questions none of them could answer alone.